In [ ]:
# Install libraries for fine-tuning
# transformers = load AI models
# peft = adds LoRA adapter for efficient fine-tuning
# trl = supervised fine-tuning trainer
# bitsandbytes = 4-bit quantization to fit model on free GPU
# datasets = load training data from HuggingFace
# accelerate = helps run model efficiently on GPU
# pandas = read and write CSV files

!pip install transformers peft trl bitsandbytes accelerate datasets pandas -q

In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
import gc

# Clear GPU memory from any previous runs
torch.cuda.empty_cache()
gc.collect()
print("Memory cleared!")

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"GPU memory free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")

# 4-bit quantization to fit model on free GPU
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

# Load Mistral 7B
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

print("Loading tokenizer")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print("Loading model in 4-bit")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)
print("Model loaded")

# Load German legal QA training data
print("Loading training data")
training_data = load_dataset("DomainLLM/german-law-qa", split="train")
print(f"Training samples: {len(training_data)}")
print("Example question:", training_data[0]['question'])
print("Example answer:", training_data[0]['answer'])

Memory cleared!
Using device: cuda
GPU memory free: 14.5 GB
Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading model in 4-bit... (2-3 minutes)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded!
Loading training data...


...

data/train-00000-of-00001.parquet:   0%|          | 0.00/7.88M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/911k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14160 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1574 [00:00<?, ? examples/s]

Training samples: 14160
Example question: Welches Ziel muss der Täter nach § 107c StGB verfolgen, damit eine Strafbarkeit eintritt?
Example answer: Der Täter muss nach § 107c StGB mit der Absicht handeln, sich oder einem anderen Kenntnis darüber zu verschaffen, wie jemand gewählt hat.


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# Prepare model for 4-bit training
model = prepare_model_for_kbit_training(model)

# Add LoRA adapter
# LoRA only trains small extra layers, not the whole model
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
print("LoRA adapter added")

# Format training examples as conversations
def format_example(example):
    text = f"[INST] Du bist ein Rechtsexperte. Beantworte die folgende Frage kurz und präzise auf Deutsch. Nenne wenn möglich den relevanten Paragraphen:\n\n{example['question']} [/INST] {example['answer']}"
    return {"text": text}

formatted_data = training_data.map(format_example)
print(f"Formatted {len(formatted_data)} training examples")

# Create trainer
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=formatted_data,
    args=SFTConfig(
        output_dir="finetuned_model",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=100,
        learning_rate=2e-4,
        bf16=True,
        logging_steps=10,
        report_to="none",
        dataset_text_field="text",
        max_grad_norm=0,
    ),
)
print("Starting fine-tuning")
trainer.train()
print("Fine-tuning done!")

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


LoRA adapter added!
Formatted 14160 training examples
Starting fine-tuning...


Step,Training Loss
10,2.182212
20,1.173734
30,0.955403
40,0.920254
50,0.914500
60,0.888957
70,0.917415
80,0.882370
90,0.869981
100,0.906172


Fine-tuning done!


In [ ]:
import pandas as pd
from google.colab import files

# Upload dataset_clean.csv
uploaded = files.upload()
questions = pd.read_csv("dataset_clean.csv")
print(f"Loaded {len(questions)} questions")

# Switch model to inference mode
model.eval()

all_answers = []

for i, row in questions.iterrows():
    question_id = row["id"]
    question_text = row["prompt"]

    # Same prompt format as training
    prompt = f"[INST] Du bist ein österreichischer Steuerexperte. Beantworte die folgende Frage kurz und präzise auf Deutsch. Nenne wenn möglich den relevanten Paragraphen (z.B. § 7 KStG):\n\n{question_text} [/INST]"

    inputs = tokenizer(prompt, return_tensors="pt",
                      max_length=512, truncation=True).to(device)

    output_tokens = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.3,
        pad_token_id=tokenizer.eos_token_id
    )

    input_length = inputs["input_ids"].shape[1]
    answer = tokenizer.decode(output_tokens[0][input_length:],
                             skip_special_tokens=True)

    all_answers.append({"id": question_id, "answer": answer})

    # Save after every question in case of disconnect
    pd.DataFrame(all_answers).to_csv("model2_finetuned_results.csv", index=False)

    if (i + 1) % 50 == 0:
        print(f"Progress: {i + 1} / {len(questions)}")

print("All done!")
print(pd.read_csv("model2_finetuned_results.csv").head(3))

Saving dataset_clean.csv to dataset_clean.csv
Loaded 643 questions
Progress: 50 / 643
Progress: 100 / 643
Progress: 150 / 643
Progress: 200 / 643
Progress: 250 / 643
Progress: 300 / 643
Progress: 350 / 643
Progress: 400 / 643
Progress: 450 / 643
Progress: 500 / 643
Progress: 550 / 643
Progress: 600 / 643
All done!
             id                                             answer
0  CORP-TAX-001  Die steuerliche Bemessungsgrundlage für die Kö...
1  CORP-TAX-002  Nach § 10 Abs. 1 des Körperschaftssteuergesetz...
2  CORP-TAX-003  Nach § 7 Abs. 1 KStG sind die Körperschaften v...


In [ ]:
from google.colab import files
files.download("model2_finetuned_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>